# ResNet Classifier - Make, Model, Year
Combining VMMRdb and StanfordCars into a single dataset to train on.

### Hyperparameters

In [25]:
# Num of images per class
MIN_CLASSES = 40
MAX_CLASSES = 100

# batch size
BATCH_SIZE = 16

# Number of iterations over dataset
epochs = 60

## Imports

In [26]:
import torchvision.transforms as tt
import torchvision.models as models
from torch.utils.data import Dataset
from torch.utils.data import ConcatDataset
from torch.utils.data import DataLoader
from torch.utils.data import Subset
from torch.utils.data import TensorDataset
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
import torch
import pennylane as qml

from collections import Counter, defaultdict
import scipy.io as sio
import pathlib
import os
import re
from PIL import Image
from tqdm import tqdm
import kagglehub
import random

## StanfordCars Setup

### Download Dataset

In [27]:
import kagglehub

# Download latest version
path_stanford_cars = kagglehub.dataset_download("eduardo4jesus/stanford-cars-dataset")

print("Path to dataset files:", path_stanford_cars)

Path to dataset files: C:\Users\godca\.cache\kagglehub\datasets\eduardo4jesus\stanford-cars-dataset\versions\1


### StanfordCars Dataset Definition
Class Definition modified from PyTorch's implementation

In [28]:
import pathlib
import scipy.io as sio
from torchvision.datasets.folder import default_loader

class StanfordCars(Dataset):
    def __init__(self, root, global_class_to_idx, transform=None, max_images_per_class=None, class_normalizer=None):
        self.root = pathlib.Path(root)
        self.class_to_idx = global_class_to_idx
        self.loader = default_loader
        self.transform = transform
        self.class_normalizer = class_normalizer
        self.class_counts = Counter()

        devkit = self.root / "car_devkit/devkit"
        ann_path = devkit / "cars_train_annos.mat"
        img_dir = self.root / "cars_train/cars_train"

        annotations = sio.loadmat(ann_path, squeeze_me=True)["annotations"]
        raw_classes = sio.loadmat(devkit / "cars_meta.mat", squeeze_me=True)["class_names"].tolist()

        self.samples = []
        for ann in annotations:
            path = str(img_dir / ann["fname"])
            cls = raw_classes[ann["class"] - 1]

            if class_normalizer:
                cls = class_normalizer(cls)

            idx = self.class_to_idx.get(cls, None)
            if idx is None:
              continue

            if max_images_per_class and self.class_counts[cls] >= max_images_per_class:
              continue

            self.samples.append((path, idx))
            self.class_counts[cls] += 1

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, y = self.samples[idx]
        img = self.loader(path)
        if self.transform:
            img = self.transform(img)
        return img, y

## VMMRdb Setup

### Dataset Download

In [29]:
path_vmmrdb = kagglehub.dataset_download("prabashwara/vmmrdb-dataset")

print("Path to dataset files:", path_vmmrdb)

Path to dataset files: C:\Users\godca\.cache\kagglehub\datasets\prabashwara\vmmrdb-dataset\versions\1


### Custom VMMRDB Dataset Definition

In [30]:
class VMMRDB_Dataset(Dataset):
    def __init__(self, root_dir, global_class_to_idx, class_normalization, max_images_per_class=None, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.class_to_idx = global_class_to_idx
        self.classes = []
        self.class_counts = Counter()

        # Images to class names
        for class_name in sorted(os.listdir(root_dir)):
          class_path = os.path.join(root_dir, class_name)
          if os.path.isdir(class_path):
            stripped = class_normalization(class_name)
            idx = self.class_to_idx.get(stripped, None)
            if idx is None:
              continue
            for img_name in os.listdir(class_path):
              if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                if max_images_per_class and self.class_counts[stripped] >= max_images_per_class:
                  continue

                self.image_paths.append(os.path.join(class_path, img_name))
                self.labels.append(idx)
                self.class_counts[stripped] += 1


        self.classes = self.class_to_idx.keys()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        return image, label

## Dataset Combination

### Normalization Function
This function is used to normalize the class labels in both the VMMRdb dataset and StanfordCars dataset

In [31]:
# Not part of model or make
EXTRA = {
    'sedan', 'coupe', 'convertible', 'hatchback', 'suv', 'wagon', 'van',
    'minivan', 'pickup', 'truck', 'cab', 'crew', 'extended', 'regular',
    'quad', 'club', 'cargo', 'passenger', 'sut', 'roadster', 'hybrid',
    'conv.', 'drophead', 'supercab', 'classic', 'superleggera',
    'abarth', 'activehybrid', 'series', 'electric',
}

# Normalizes classes in both datasets
def normalize_classes(car_class):
  car_class = re.sub('_', ' ', car_class).lower()
  car_class = re.sub("-", "", car_class)
  car_class_components = car_class.split(" ")
  car_class_components = [t for t in car_class_components if t not in EXTRA]
  car_class_components = ["mercedes benz" if t == "mercedesbenz" else t for t in car_class_components]
  car_class_components = ["rolls royce" if t == "rollsroyce" else t for t in car_class_components]
  return " ".join(car_class_components)

### Combining the two datasets
Using a minimum number of 40 images and a maximum number of 100 per class.

In [32]:
# Building global class_to_idx so we can combine the two datasets properly

def build_global_class_to_idx(stanford_root, vmmrdb_root, normalize_classes, min_classes):
    class_counts = Counter()


    # Stanford Cars
    devkit = pathlib.Path(stanford_root) / "car_devkit/devkit"
    ann_path = devkit / "cars_train_annos.mat"
    annotations = sio.loadmat(ann_path, squeeze_me=True)["annotations"]
    raw_classes = sio.loadmat(devkit / "cars_meta.mat", squeeze_me=True)["class_names"].tolist()

    # Normalizing class names + retrieving image count per class
    for ann in annotations:
        cls = normalize_classes(raw_classes[ann["class"] - 1])
        class_counts[cls] += 1

    # VMMRdb
    for class_name in sorted(os.listdir(vmmrdb_root)):
        class_path = os.path.join(vmmrdb_root, class_name)
        if os.path.isdir(class_path):

            # Normalizing class names + retrieving image count per class
            cls = normalize_classes(class_name)
            for img in os.listdir(class_path):
                if img.lower().endswith((".jpg", ".jpeg", ".png")):
                    class_counts[cls] += 1

    # Removing classes with less than min_classes number of images
    trimmed_classes = {cls for cls, count in class_counts.items() if count >= min_classes}
    class_to_idx = {cls : idx for idx, cls in enumerate(sorted(trimmed_classes))}
    class_counts = {cls : count for cls,count in class_counts.items() if cls in trimmed_classes}

    return class_to_idx, class_counts

# Building the global class to idx and finding the number of images per class + removes classes with less than MIN_CLASSES images
global_class_to_idx, class_counts = build_global_class_to_idx(path_stanford_cars, path_vmmrdb, normalize_classes, MIN_CLASSES)

### Defining Training and Test Transforms

Run the following cell if MIN_CLASSES or MAX_CLASSES is modified or the dataset is modified to get the new pixel mean and std. Then replace mean and std in the cell following this one with the new mean and std.

In [33]:
transform = tt.Compose([
    tt.Resize((224,224)),
    tt.ToTensor(),
])
# Full dataset used to calculate pixel mean and std
stanford_dataset = StanfordCars(
    root=path_stanford_cars,
    global_class_to_idx=global_class_to_idx,
    class_normalizer=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=transform
)
vmmrdb_dataset = VMMRDB_Dataset(
    root_dir=path_vmmrdb,
    global_class_to_idx=global_class_to_idx,
    class_normalization=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=transform
)
full_dataset = ConcatDataset([stanford_dataset, vmmrdb_dataset])
data_loader = DataLoader(full_dataset, BATCH_SIZE, num_workers = 0, pin_memory = True, shuffle=False)


In [34]:
# Mean and Standard Deviation Calculation
mean = 0
ex2 = 0
total_pixels = 0

loop = tqdm(data_loader)
for images, _ in loop:
    B, C, H, W = images.shape

    images = images.view(B, C, -1) # H x W

    mean += images.sum(dim=(0, 2))
    ex2 += (images ** 2).sum(dim=(0, 2))

    total_pixels += B * H * W

    loop.set_description("mean + std calculation")

mean /= total_pixels
ex2 /= total_pixels

std = torch.sqrt(ex2 - mean ** 2)

print()
print("Mean:", mean)
print("Std:", std)

mean + std calculation:   0%|          | 14/9659 [00:01<16:11,  9.92it/s]


KeyboardInterrupt: 

In [11]:
# mean = [0.4369, 0.4330, 0.4285]
# std = [0.2681, 0.2649, 0.2673]

# hqnn
# Mean: tensor([0.4380, 0.4347, 0.4282])
# Std: tensor([0.2681, 0.2645, 0.2674])

# # Defines image preprocessing for the training and test datasets
# train_transform = tt.Compose([
#     tt.Resize((224,224)),

#     # Random Transformations
#     tt.RandomHorizontalFlip(),
#     tt.ColorJitter(
#         brightness=0.2,
#         contrast=0.2,
#         saturation=0.2,
#         hue=0.1
#     ),
#     tt.RandomRotation(100),

#     tt.ToTensor(),
#     tt.Normalize(mean, std)
# ])

# test_transform = tt.Compose([
#     tt.Resize((224,224)),
#     tt.ToTensor(),
#     tt.Normalize(mean, std)
# ])

In [35]:
# trying this from my previous implementation -- olivia 
# stats = ((0.4708, 0.4602, 0.4550),(0.2891, 0.2882, 0.2967)) # mean for RGB, standard deviation for RGB

mean = [0.4380, 0.4347, 0.4282]
std = [0.2681, 0.2645, 0.2674]
stats = (mean, std)

train_transform = tt.Compose([
    tt.Resize((224, 224)),
    tt.RandomHorizontalFlip(),
    tt.RandomRotation(15),
    tt.ToTensor(),  
    tt.Normalize(*stats)
])

test_transform = tt.Compose([
    tt.Resize((224, 224)),
    tt.ToTensor(),
    tt.Normalize(*stats)
])

### Defining Datasets For Training and Testing

In [36]:
# Need to create the dataset objects
# twice to properly create the test and training sets with the correct
# transforms tied to them

# Test dataset
stanford_test = StanfordCars(
    root=path_stanford_cars,
    global_class_to_idx=global_class_to_idx,
    class_normalizer=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=test_transform
)
vmmrdb_test = VMMRDB_Dataset(
    root_dir=path_vmmrdb,
    global_class_to_idx=global_class_to_idx,
    class_normalization=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=test_transform
)

# Train dataset
stanford_train = StanfordCars(
    root=path_stanford_cars,
    global_class_to_idx=global_class_to_idx,
    class_normalizer=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=train_transform
)
vmmrdb_train = VMMRDB_Dataset(
    root_dir=path_vmmrdb,
    global_class_to_idx=global_class_to_idx,
    class_normalization=normalize_classes,
    max_images_per_class = MAX_CLASSES,
    transform=train_transform
)

### Splitting the Training and Testing datasets

In [37]:
# Splitting into training and testing
full = ConcatDataset([stanford_train, vmmrdb_train])

# 70 - 10 - 20 split
train_size = int(0.70 * len(full))
val_size = int(0.10 * len(full))
test_size = len(full) - val_size - train_size

# Test + train + val set
generator = torch.Generator().manual_seed(0)
indices = torch.randperm(len(full), generator=generator)
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]
test_indices = indices[train_size + val_size:]

dataset_train = Subset(ConcatDataset([stanford_train, vmmrdb_train]), train_indices)
dataset_val = Subset(ConcatDataset([stanford_test, vmmrdb_test]), val_indices)
dataset_test = Subset(ConcatDataset([stanford_test, vmmrdb_test]), test_indices)

In [38]:
train_loader = DataLoader(dataset_train, BATCH_SIZE, num_workers = 0, pin_memory = True, shuffle=True)
val_loader = DataLoader(dataset_val, BATCH_SIZE, num_workers = 0, pin_memory = True, shuffle=False)
test_loader = DataLoader(dataset_test, BATCH_SIZE, num_workers = 0, pin_memory = True, shuffle=False)


### ResNet-50
Using the ResNet-50 Architecture to train a model that predicts model, make and year

In [39]:
class Bottleneck(nn.Module):
  expansion = 4
  def __init__(self, in_channels, out_channels, stride=1):
    super(Bottleneck, self).__init__()

    # 1x1 conv to reduce channels
    self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1, padding=0, bias=False)
    self.batch_norm1 = nn.BatchNorm2d(out_channels)

    # 3x3 conv to process on reduced channels
    self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
    self.batch_norm2 = nn.BatchNorm2d(out_channels)

    # 1x1 conv to expand channels
    self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, stride=1, padding=0, bias=False)
    self.batch_norm3 = nn.BatchNorm2d(out_channels * self.expansion)

    # Skip Connection
    self.shortcut = nn.Sequential()

    if stride != 1 or in_channels != out_channels * self.expansion:
      self.shortcut = nn.Sequential(
          nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channels * self.expansion)
      )

  def forward(self, x):

    # Channel reduction
    out = F.relu(self.batch_norm1(self.conv1(x)))
    # Process
    out = F.relu(self.batch_norm2(self.conv2(out)))
    # Channel expansion
    out = self.batch_norm3(self.conv3(out))
    # Residual connection
    out += self.shortcut(x)
    return F.relu(out)

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super(ResNet, self).__init__()

        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2,
                               padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # ResNet layers
        self.layer1 = self._make_layer(block, 64, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride):
        layers = []

        # First block
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels * block.expansion

        # Remaining blocks
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        # First convolution step using 7x7
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        # ResNet Layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Fully connected portion
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x



def ResNet50(num_classes):
    return ResNet(Bottleneck, [3, 4, 6, 3], num_classes)

## Training Code

In [ ]:
# HQNN training on top of frozen ResNet-50 backbone
# pretrained model: ResNet-50 with weights trained on ImageNet-1k

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
num_classes = len(global_class_to_idx)
loss_criterion = nn.CrossEntropyLoss()


class HybridHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int, n_qubits: int = 6, n_layers: int = 2, adapter_dim: int = 128):
        super().__init__()
        self.adapter = nn.Sequential(
            nn.Linear(in_dim, adapter_dim),
            nn.ReLU(inplace=True),
            nn.Linear(adapter_dim, n_qubits),
        )
        self.scale = nn.Parameter(torch.ones(n_qubits))

        dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="X")
            qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.qlayer = qml.qnn.TorchLayer(circuit, {"weights": (n_layers, n_qubits, 3)})
        self.classifier = nn.Linear(n_qubits, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.adapter(x)
        angles = torch.tanh(h) * self.scale * torch.pi
        q_out = self.qlayer(angles)
        return self.classifier(q_out)


net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2).to(device)
feature_dim = net.fc.in_features

# Freeze the backbone and replace final layer with identity for feature extraction
backbone_fc = net.fc
net.fc = nn.Identity()
for p in net.parameters():
    p.requires_grad = False

hqnn_head = HybridHead(in_dim=feature_dim, num_classes=num_classes, n_qubits=6, n_layers=2, adapter_dim=128).to(device)
optimizer = optim.AdamW(hqnn_head.parameters(), lr=1e-3, weight_decay=1e-4)


@torch.no_grad()
def cache_features(loader, model):
    model.eval()
    feats = []
    labels = []
    for X, y in tqdm(loader, leave=False, desc="Caching feats"):
        X = X.to(device, non_blocking=True)
        f = model(X).float().cpu()
        feats.append(f)
        labels.append(y.cpu())
    return torch.cat(feats, dim=0), torch.cat(labels, dim=0)


print("Caching train/val features from frozen ResNet backbone...")
train_feats, train_labels = cache_features(train_loader, net)
val_feats, val_labels = cache_features(val_loader, net)

hqnn_batch_size = 64
train_feat_loader = DataLoader(TensorDataset(train_feats, train_labels), batch_size=hqnn_batch_size, shuffle=True)
val_feat_loader = DataLoader(TensorDataset(val_feats, val_labels), batch_size=hqnn_batch_size, shuffle=False)

checkpoint_path = "./hqnn_checkpoint.pth"
best_checkpoint_path = "./hqnn_best_checkpoint.pth"
start_epoch = 0

train_losses = []
val_losses = []
val_accuracies = []
best_val_acc = 0.0

try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    hqnn_head.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    val_accuracies = checkpoint.get("val_accuracies", [])
    best_val_acc = checkpoint.get("best_val_acc", 0.0)
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resuming HQNN from epoch {start_epoch}")
except FileNotFoundError:
    print("No HQNN checkpoint found, starting fresh.")

hqnn_epochs = min(epochs, 20)

for epoch in range(start_epoch, hqnn_epochs):
    hqnn_head.train()
    running_loss = 0.0
    num_batches = 0

    train_loop = tqdm(train_feat_loader, leave=True)
    for Xf, y in train_loop:
        Xf = Xf.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        output = hqnn_head(Xf)
        loss = loss_criterion(output, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1
        train_loop.set_description(f"HQNN Epoch [{epoch+1}/{hqnn_epochs}]")
        train_loop.set_postfix(loss=loss.item())

    train_loss = running_loss / max(num_batches, 1)
    train_losses.append(train_loss)

    hqnn_head.eval()
    val_loss = 0.0
    val_batches = 0
    correct = 0
    total = 0

    with torch.no_grad():
        val_loop = tqdm(val_feat_loader, leave=True, desc="HQNN Validation")
        for Xf, y in val_loop:
            Xf = Xf.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            output = hqnn_head(Xf)
            loss = loss_criterion(output, y)

            val_loss += loss.item()
            val_batches += 1
            pred = output.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    val_loss /= max(val_batches, 1)
    val_acc = correct / max(total, 1)

    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(f"HQNN epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

    checkpoint = {
        "epoch": epoch,
        "model_state": hqnn_head.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_accuracies": val_accuracies,
        "best_val_acc": best_val_acc,
    }
    torch.save(checkpoint, checkpoint_path)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        checkpoint["best_val_acc"] = best_val_acc
        torch.save(checkpoint, best_checkpoint_path)

print(f"Finished HQNN training. Best val accuracy: {best_val_acc:.4f}")

# Restore classical head
net.fc = backbone_fc

device: cuda
Caching train/val features from frozen ResNet backbone...


No HQNN checkpoint found, starting fresh.


HQNN Validation: 100%|██████████| 242/242 [00:05<00:00, 40.94it/s]


HQNN epoch 1: train_loss=7.4961, val_loss=7.3285, val_acc=0.0012


HQNN Validation: 100%|██████████| 242/242 [00:05<00:00, 46.04it/s]


...

HQNN epoch 18: train_loss=6.1368, val_loss=6.3969, val_acc=0.0057


HQNN Validation: 100%|██████████| 242/242 [00:04<00:00, 57.15it/s]


HQNN epoch 19: train_loss=6.1208, val_loss=6.3986, val_acc=0.0052


HQNN Validation: 100%|██████████| 242/242 [00:04<00:00, 56.52it/s]

HQNN epoch 20: train_loss=6.1064, val_loss=6.4021, val_acc=0.0050
Finished HQNN training. Best val accuracy: 0.0061


In [ ]:
# HQNN fine-tuning with partial backbone unfreezing + top-5 metrics
# Trains end-to-end (no cached features) so unfrozen layers can actually learn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
num_classes = len(global_class_to_idx)
loss_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)


class HybridHead(nn.Module):
    def __init__(self, in_dim: int, num_classes: int, n_qubits: int = 6, n_layers: int = 2, adapter_dim: int = 128):
        super().__init__()
        self.adapter = nn.Sequential(
            nn.Linear(in_dim, adapter_dim),
            nn.ReLU(inplace=True),
            nn.Linear(adapter_dim, n_qubits),
        )
        self.scale = nn.Parameter(torch.ones(n_qubits))

        dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(inputs, weights):
            qml.AngleEmbedding(inputs, wires=range(n_qubits), rotation="X")
            qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.qlayer = qml.qnn.TorchLayer(circuit, {"weights": (n_layers, n_qubits, 3)})
        self.classifier = nn.Linear(n_qubits, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.adapter(x)
        angles = torch.tanh(h) * self.scale * torch.pi
        q_out = self.qlayer(angles)
        return self.classifier(q_out)


# Build pretrained backbone and expose penultimate features.
net = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2).to(device)
feature_dim = net.fc.in_features
net.fc = nn.Identity()

# Freeze everything, then unfreeze only layer4 for partial fine-tuning.
for p in net.parameters():
    p.requires_grad = False
for p in net.layer4.parameters():
    p.requires_grad = True

hqnn_head = HybridHead(in_dim=feature_dim, num_classes=num_classes, n_qubits=6, n_layers=2, adapter_dim=128).to(device)

# Different learning rates for backbone vs HQNN head.
optimizer = optim.AdamW(
    [
        {"params": net.layer4.parameters(), "lr": 1e-5, "weight_decay": 1e-4},
        {"params": hqnn_head.parameters(), "lr": 1e-3, "weight_decay": 1e-4},
    ]
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=min(epochs, 20))

checkpoint_path = "./hqnn_unfreeze_checkpoint.pth"
best_checkpoint_path = "./hqnn_unfreeze_best_checkpoint.pth"
start_epoch = 0

train_losses = []
val_losses = []
val_top1_accuracies = []
val_top5_accuracies = []
best_val_top1 = 0.0

try:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    net.load_state_dict(checkpoint["backbone_state"])
    hqnn_head.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    train_losses = checkpoint.get("train_losses", [])
    val_losses = checkpoint.get("val_losses", [])
    val_top1_accuracies = checkpoint.get("val_top1_accuracies", [])
    val_top5_accuracies = checkpoint.get("val_top5_accuracies", [])
    best_val_top1 = checkpoint.get("best_val_top1", 0.0)
    start_epoch = checkpoint["epoch"] + 1
    print(f"Resuming HQNN+unfreeze from epoch {start_epoch}")
except FileNotFoundError:
    print("No HQNN+unfreeze checkpoint found, starting fresh.")

hqnn_epochs = min(epochs, 20)

for epoch in range(start_epoch, hqnn_epochs):
    net.train()
    hqnn_head.train()

    running_loss = 0.0
    num_batches = 0

    train_loop = tqdm(train_loader, leave=True)
    for X, y in train_loop:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        feats = net(X)
        output = hqnn_head(feats)
        loss = loss_criterion(output, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        num_batches += 1
        train_loop.set_description(f"HQNN+FT Epoch [{epoch+1}/{hqnn_epochs}]")
        train_loop.set_postfix(loss=loss.item())

    train_loss = running_loss / max(num_batches, 1)
    train_losses.append(train_loss)

    net.eval()
    hqnn_head.eval()

    val_loss = 0.0
    val_batches = 0
    top1_correct = 0
    top5_correct = 0
    total = 0

    with torch.no_grad():
        val_loop = tqdm(val_loader, leave=True, desc="HQNN+FT Validation")
        for X, y in val_loop:
            X = X.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            feats = net(X)
            output = hqnn_head(feats)
            loss = loss_criterion(output, y)

            val_loss += loss.item()
            val_batches += 1

            pred_top1 = output.argmax(dim=1)
            top1_correct += (pred_top1 == y).sum().item()

            topk_idx = output.topk(k=5, dim=1).indices
            top5_correct += topk_idx.eq(y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    val_loss /= max(val_batches, 1)
    val_top1 = top1_correct / max(total, 1)
    val_top5 = top5_correct / max(total, 1)

    val_losses.append(val_loss)
    val_top1_accuracies.append(val_top1)
    val_top5_accuracies.append(val_top5)

    print(
        f"HQNN+FT epoch {epoch+1}: "
        f"train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, "
        f"val_top1={val_top1:.4f}, val_top5={val_top5:.4f}"
    )

    scheduler.step()

    checkpoint = {
        "epoch": epoch,
        "backbone_state": net.state_dict(),
        "model_state": hqnn_head.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "val_top1_accuracies": val_top1_accuracies,
        "val_top5_accuracies": val_top5_accuracies,
        "best_val_top1": best_val_top1,
    }
    torch.save(checkpoint, checkpoint_path)

    if val_top1 > best_val_top1:
        best_val_top1 = val_top1
        checkpoint["best_val_top1"] = best_val_top1
        torch.save(checkpoint, best_checkpoint_path)

print(f"Finished HQNN+FT training. Best val top-1 accuracy: {best_val_top1:.4f}")

device: cuda
No HQNN+unfreeze checkpoint found, starting fresh.


HQNN+FT Validation: 100%|██████████| 966/966 [01:06<00:00, 14.52it/s]


HQNN+FT epoch 1: train_loss=7.3942, val_loss=7.0705, val_top1=0.0019, val_top5=0.0136


HQNN+FT Validation: 100%|██████████| 966/966 [01:01<00:00, 15.82it/s]


HQNN+FT epoch 2: train_loss=6.9821, val_loss=6.8734, val_top1=0.0038, val_top5=0.0162


...

HQNN+FT epoch 18: train_loss=6.2691, val_loss=6.3240, val_top1=0.0101, val_top5=0.0421


HQNN+FT Validation: 100%|██████████| 966/966 [01:04<00:00, 15.00it/s]


HQNN+FT epoch 19: train_loss=6.2661, val_loss=6.3248, val_top1=0.0102, val_top5=0.0412


HQNN+FT Validation: 100%|██████████| 966/966 [01:04<00:00, 14.88it/s]


HQNN+FT epoch 20: train_loss=6.2643, val_loss=6.3259, val_top1=0.0102, val_top5=0.0417
Finished HQNN+FT training. Best val top-1 accuracy: 0.0102
